In [1]:
import pandas as pd
import pyreadstat
import matplotlib.pyplot as plt
import scipy.stats as sts
import seaborn as sns
import numpy as np
from scipy.stats import f
from scipy.stats import t
from statsmodels.stats.diagnostic import het_white
from statsmodels.stats.diagnostic import het_breuschpagan
from scipy.stats import zscore
import statsmodels.api as sm
from scipy.stats import chi2
from scipy.stats import boxcox_llf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.diagnostic import linear_reset
from statsmodels.stats.diagnostic import linear_harvey_collier
from statsmodels.sandbox.regression.gmm import IV2SLS
from statsmodels.stats.moment_helpers import cov2corr
from scipy.stats import norm
from statsmodels.miscmodels.ordinal_model import OrderedModel
import warnings 
from statsmodels.formula.api import logit
from scipy.optimize import minimize

In [2]:
warnings.filterwarnings('ignore')

In [3]:
data = pd.read_csv('drom_cleaned.csv')
data

,Стоимость,Модель,Город,Пробег,Объём двигателя,Тип топлива,Возраст мотоцикла,Коробка (код),Тактность (код),Подача топлива (код),...,Бренд_Harley-Davidson,Бренд_Honda,Бренд_Indian,Бренд_Kawasaki,Бренд_MV Agusta,Бренд_Royal Enfield,Бренд_Suzuki,Бренд_Triumph,Бренд_Yamaha,Бренд_Другое
0,12000.0,Иж 5,Курья,19000.0,350,бензин,27,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,12500.0,Kayo Basic YX125,Набережные Челны,500.0,125,бензин,0,1,1,0,...,0,0,0,0,0,0,0,0,0,1
2,15000.0,Иж Планета 4,Зерноград,999.0,350,бензин,26,0,1,0,...,0,0,0,0,0,0,0,0,0,1
3,15000.0,Урал 5557,Глазов,20000.0,620,бензин,38,0,1,0,...,0,0,0,0,0,0,0,0,0,1
4,15000.0,Восход 2М,Магнитогорск,10000.0,175,бензин,47,0,1,0,...,0,0,0,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8302,7650000.0,Harley-Davidson Tri Glide Ultra FLHTCUTG,Новосибирск,0.0,1868,бензин,1,0,1,1,...,1,0,0,0,0,0,0,0,0,0
8303,7950000.0,Harley-Davidson CVO Road Glide,Новосибирск,0.0,1983,бензин,1,0,1,1,...,1,0,0,0,0,0,0,0,0,0
8304,8250000.0,Harley-Davidson CVO Road Glide,Новосибирск,0.0,1977,бензин,0,0,1,1,...,1,0,0,0,0,0,0,0,0,0
8305,8500000.0,Harley-Davidson CVO Road Glide,Красноярск,0.0,1923,бензин,2,0,1,1,...,1,0,0,0,0,0,0,0,0,0


In [5]:
data['Стоимость'].median()

340000.0

In [6]:
data['Стоимость'].mean()

499540.1050319008

Создадим переменную Стоимость_грань, равную 1, если Стоимость выше 500 тысяч рублей и 0 в противном случае.

In [8]:
data['Стоимость_грань'] = np.where(data['Стоимость'] > 500000, 1, 0)
data

,Стоимость,Модель,Город,Пробег,Объём двигателя,Тип топлива,Возраст мотоцикла,Коробка (код),Тактность (код),Подача топлива (код),...,Бренд_Honda,Бренд_Indian,Бренд_Kawasaki,Бренд_MV Agusta,Бренд_Royal Enfield,Бренд_Suzuki,Бренд_Triumph,Бренд_Yamaha,Бренд_Другое,Стоимость_грань
0,12000.0,Иж 5,Курья,19000.0,350,бензин,27,0,0,0,...,0,0,0,0,0,0,0,0,1,0
1,12500.0,Kayo Basic YX125,Набережные Челны,500.0,125,бензин,0,1,1,0,...,0,0,0,0,0,0,0,0,1,0
2,15000.0,Иж Планета 4,Зерноград,999.0,350,бензин,26,0,1,0,...,0,0,0,0,0,0,0,0,1,0
3,15000.0,Урал 5557,Глазов,20000.0,620,бензин,38,0,1,0,...,0,0,0,0,0,0,0,0,1,0
4,15000.0,Восход 2М,Магнитогорск,10000.0,175,бензин,47,0,1,0,...,0,0,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8302,7650000.0,Harley-Davidson Tri Glide Ultra FLHTCUTG,Новосибирск,0.0,1868,бензин,1,0,1,1,...,0,0,0,0,0,0,0,0,0,1
8303,7950000.0,Harley-Davidson CVO Road Glide,Новосибирск,0.0,1983,бензин,1,0,1,1,...,0,0,0,0,0,0,0,0,0,1
8304,8250000.0,Harley-Davidson CVO Road Glide,Новосибирск,0.0,1977,бензин,0,0,1,1,...,0,0,0,0,0,0,0,0,0,1
8305,8500000.0,Harley-Davidson CVO Road Glide,Красноярск,0.0,1923,бензин,2,0,1,1,...,0,0,0,0,0,0,0,0,0,1


Оценим logit–модель бинарного выбора для переменной Стоимость_грань. 
В качестве объясняющих факторов используем следующие показатели: пробег, объём двигателя, возраст мотоцикла, коробка (код), тактность (код), подача топлива (код).

Оцениваем следующую модель:
    
$$logit(P(\text {Стоимость_грань} = 1)) = \beta_0 + \beta_1 * Пробег + \beta_2 * \text {Объём двигателя} + \beta_3 * \text{Возраст мотоцикла} + \beta_4 * Коробка(код) + \beta_5 * Тактность(код) + \beta_6 * \text{Подача топлива(код)}$$

In [11]:
param = ['Пробег', 'Объём двигателя', 'Возраст мотоцикла', 'Коробка (код)', 'Тактность (код)', 'Подача топлива (код)']

X = data[param]
X = sm.add_constant(X)
y = data['Стоимость_грань']

logit_model = sm.Logit(y, X).fit(disp=False)
logit_model.summary2()

<class 'statsmodels.iolib.summary2.Summary'>
"""
                           Results: Logit
=====================================================================
Model:                 Logit             Pseudo R-squared:  0.631    
Dependent Variable:    Стоимость_грань   AIC:               3944.2823
Date:                  2025-05-08 23:07  BIC:               3993.4563
No. Observations:      8307              Log-Likelihood:    -1965.1  
Df Model:              6                 LL-Null:           -5324.2  
Df Residuals:          8300              LLR p-value:       0.0000   
Converged:             1.0000            Scale:             1.0000   
No. Iterations:        8.0000                                        
---------------------------------------------------------------------
                      Coef.  Std.Err.    z     P>|z|   [0.025  0.975]
---------------------------------------------------------------------
const                -2.3373   0.1944 -12.0244 0.0000 -2.7183 -1.9563
Пробег               -0.0000   0.0000  -6.7193 0.0000 -0.0000 -0.0000
Объём двигателя       0.0060   0.0002  31.8074 0.0000  0.0056  0.0064
Возраст мотоцикла    -0.1110   0.0076 -14.5741 0.0000 -0.1259 -0.0960
Коробка (код)         0.3364   0.2532   1.3288 0.1839 -0.1598  0.8327
Тактность (код)      -2.4719   0.2076 -11.9055 0.0000 -2.8788 -2.0649
Подача топлива (код)  3.3220   0.0993  33.4473 0.0000  3.1274  3.5167
=====================================================================

"""

Получаем:
    
$$logit(P(\text {Стоимость_грань} = 1)) = -2.33 + 0.006 * \text {Объём двигателя} - 0.11 * \text{Возраст мотоцикла} + 0.34 * Коробка(код) - 2.47 * Тактность(код) + 3.32 * \text{Подача топлива(код)}$$

Все переменные, кроме Коробка (код), статистически значимы при 1% уровне.

Построим прогноз вероятности, что мотоцикл с заданными характеристиками будет выставлен по цене ниже 500 тысяч рублей

In [14]:
p_cr = logit_model.predict(X)

p_no_cr = 1 - p_cr

data['p_under_500'] = p_no_cr
data['p_under_500']

0       0.972248
1       0.976639
2       0.996375
3       0.996493
4       0.999893
          ...   
8302    0.000069
8303    0.000035
8304    0.000032
8305    0.000055
8306    0.004811
Name: p_under_500, Length: 8307, dtype: float64

Оценим средние предельные эффекты для вероятности быть арестованным по всем факторам.

Для непрерывных переменных, то есть: пробег, объём двигателя и возраст мотоцикла средние предельные эффекты будет рассчитывать по следующей формуле:

$$
\frac{\partial \mathbb{P}(y=1 \mid x)}{\partial x_j} = \hat{\beta}_j \cdot \phi(x\hat{\beta}) = \hat{\beta}_j \cdot \hat{P}(1 - \hat{P})
$$

Для бинарных переменных коробка передач, тактность и подача топлива - по формуле разности условных вероятностей:
    
$$
\Delta_j = \mathbb{P}(y=1 \mid x_j = 1, \bar{x}_{-j}) - \mathbb{P}(y=1 \mid x_j = 0, \bar{x}_{-j})
$$

In [15]:
X_mean = X.mean()

xb = np.dot(X_mean, logit_model.params)

p = 1 / (1 + np.exp(-xb))

Q_c = logit_model.params * p * (1 - p)

bi = ['Коробка (код)', 'Тактность (код)', 'Подача топлива (код)']
Q_bi = {}

for i in bi:
    X_1, X_0 = X_mean.copy(), X_mean.copy()
    X_1[i], X_0[i] = 1, 0

    p1 = 1 / (1 + np.exp(-np.dot(X_1, logit_model.params)))
    p0 = 1 / (1 + np.exp(-np.dot(X_0, logit_model.params)))
    Q_bi[i] = p1 - p0

Q = Q_c.to_dict()
Q.update(Q_bi)

Q

{'const': -0.3668654223977183,
 'Пробег': -2.6247678638272008e-06,
 'Объём двигателя': 0.0009393291572258078,
 'Возраст мотоцикла': -0.017415627537059504,
 'Коробка (код)': 0.05789238808696853,
 'Тактность (код)': -0.5407632723357481,
 'Подача топлива (код)': 0.5641135137265806}

Оценим отношение шансов для всех факторов и проверим их значимость.

In [16]:
summary = pd.DataFrame({
    'Коэффициент': logit_model.params,
    'Отношение шансов(odds ratio)': np.exp(logit_model.params),
    'p-value': logit_model.pvalues
})

summary

,Коэффициент,Отношение шансов(odds ratio),p-value
const,-2.337328,0.096585,2.644495e-33
Пробег,-0.000017,0.999983,1.826273e-11
Объём двигателя,0.005985,1.006002,5.106649e-222
Возраст мотоцикла,-0.110956,0.894978,4.104461e-48
Коробка (код),0.336437,1.399950,1.839219e-01
Тактность (код),-2.471864,0.084427,1.107754e-32
Подача топлива (код),3.322029,27.716536,2.816377e-245
